# Step 6: generate MBBQ model outputs on Colab

Runs `my_research/generation/run_step6.py` for one model over en, nl, es, tr x {main, control}, with the paper-era generation settings from `my_research/generation/ask_model_paper.py` (the authors' `models.py` at commit `20ae39b`): Mistral gets 1000 new tokens and batch size 16, Zephyr 256 and 32, each with its original prompt format and a pinned model revision.
Everything is written to Google Drive, and finished subsets are skipped, so **after a disconnect just run all cells again**.

**Russian:** the last five sections (pilot, full run, feminine variant, replication models, batch-size noise floor) are run on their own: the settings and setup cells, then that section. Each says how.

Before the first run:
1. Runtime > Change runtime type > **A100** (L4 also works). A T4 lacks bf16 and the script refuses it.
2. Optional: add a Colab secret (key icon on the left) named `HF_TOKEN` holding a read token, with notebook access on. Neither model is gated; a token only avoids anonymous rate limits.
3. Push `my_research/` to the fork (cell 2 clones it from GitHub).

Outputs go to `MyDrive/mbbq_paper`. Earlier runs in `MyDrive/mbbq` used upstream `ask_model` (100 new tokens for every model) and are not reused; the script stops if it finds outputs made with other settings in its work folder. You can delete `MyDrive/mbbq`.

In [ ]:
# Which model: run once with "mistral", then once with "zephyr".
# The replication section uses "qwen" or "tlite"; the fifth-model section "aya" or "llama2".
# Model id, pinned revision, prompt format, max_new_tokens and batch size
# come from MODELS in my_research/generation/ask_model_paper.py.
MODEL_KEY = "mistral"
DRIVE = "/content/drive/MyDrive/mbbq_paper"
REPO = "https://github.com/michaellarionov/MBBQ_Russian.git"
# requirements.txt version for the paper models; the replication models' tokenizer
# files need a newer tokenizers (DECISIONS.md). run_step6.py checks the version.
TRANSFORMERS_PIN = "4.46.3" if MODEL_KEY in ("qwen", "tlite") else "4.39.3"

In [ ]:
import os
from google.colab import drive, userdata
drive.mount("/content/drive")
try:  # optional; mbbq.py's login is skipped and the hub reads HF_TOKEN
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except userdata.SecretNotFoundError:
    pass
if not os.path.isdir("/content/MBBQ_Russian"):
    !git clone -q {REPO} /content/MBBQ_Russian
%cd /content/MBBQ_Russian
!git pull -q && git log --oneline -1
if TRANSFORMERS_PIN:
    !pip install -q transformers=={TRANSFORMERS_PIN}
!python -c "import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.get_device_name())"

## Smoke test (do this first)

One subset (Gender_identity, English, main set: 2,720 prompts). It checks that loading, generation and answer detection work, and it estimates the GPU time for the full run. Mistral's 1000-token limit makes it much slower per prompt than upstream's 100.

In [ ]:
import time
t0 = time.time()
!python my_research/generation/run_step6.py --model {MODEL_KEY} --key {MODEL_KEY}_smoke \
    --langs en --subsets Gender_identity --no_control \
    --work_dir {DRIVE}/work --results_dir {DRIVE}/smoke
smoke_secs = time.time() - t0
# `!` does not stop "Run all" on failure; this does.
assert globals().get("_exit_code", 0) == 0, "run failed: see the error above"


In [ ]:
import sys
sys.path.insert(0, "my_research/scorer")
from mbbq_scorer import annotate, attach_polarity, load_outputs, score

smoke_pkl = f"{DRIVE}/smoke/{MODEL_KEY}_smoke_en.pkl"
assert os.path.exists(smoke_pkl), "no smoke output: the run in the cell above failed"
smoke = load_outputs(smoke_pkl)
print(smoke[["prompt_id", "answer", "answer_detected"]].sample(5, random_state=0).to_string())
s = score(annotate(attach_polarity(smoke, "data", "en")))
print({k: round(v, 3) for k, v in s.items()})

# Wall time of the cell above, so it includes model download and loading:
# an overestimate. If the smoke output already existed, nothing ran.
full = 8 * 50_400  # prompts: 4 languages x {main, control}
if smoke_secs < 60:
    print("Smoke run was skipped (output already exists): no timing.")
else:
    print(f"{smoke_secs / len(smoke):.3f} s/prompt -> "
          f"~{smoke_secs / len(smoke) * full / 3600:.0f} GPU-hours for the full run")

Continue only if the answers look sensible, few are undetected, and the GPU-hour estimate fits your compute units.

## Full run

This takes many hours. If Colab disconnects, run all cells again, skipping the smoke-test cells; it resumes at the unfinished subset.

In [ ]:
!python my_research/generation/run_step6.py --model {MODEL_KEY} \
    --work_dir {DRIVE}/work --results_dir {DRIVE}/results
# `!` does not stop "Run all" on failure; this does.
assert globals().get("_exit_code", 0) == 0, "run failed: see the error above"


In [ ]:
import glob
done = sorted(os.path.basename(p) for p in glob.glob(f"{DRIVE}/results/{MODEL_KEY}_*.pkl"))
print(f"{len(done)}/8 finished:", done)

When all 8 `<model>_<lang>[_control].pkl` files are in `MyDrive/mbbq_paper/results`, download them into `my_research/results/` locally (about 50 MB per pickle) and run step 7. Record the settings and library versions (printed at the start of the run) in `my_research/README.md`.

## Russian pilot (step 7 in `NEXT_STEPS.md`)

Mistral on Gender_identity in Russian, main set only: 2,720 prompts, as many as the English smoke test. Russian text takes more tokens per word than English, so expect it to run longer than the smoke test.

Run the first two cells (settings and setup), then the two cells below; skip the smoke test and the full run.

The Russian data isn't in the repo yet (it is generated into `data/` after the stereotype vote), so the first cell generates Gender_identity into Drive, not into the repo clone. The generator is deterministic and the vote only removes templates, so every kept template gets exactly these items in the full run.

When it finishes, download `ru_pilot/<model>_pilot_ru.pkl` from Drive into `my_research/results/` for the analysis: the undetected answers and the detection phrases to add. Detection reruns offline on that pickle, so the phrase lists can still change.

In [ ]:
# Generate the Russian Gender_identity data into Drive, then ask the model.
# run_step6 uses PILOT/work/data because it exists (it links data/ only when missing).
import time
PILOT = f"{DRIVE}/ru_pilot"
PILOT_DATA = f"{PILOT}/work/data"
os.makedirs(PILOT_DATA, exist_ok=True)
!python my_research/templates/generate_ru.py --category Gender_identity --out {PILOT_DATA}
assert globals().get("_exit_code", 0) == 0, "generation failed: see the error above"

t0 = time.time()
!python my_research/generation/run_step6.py --model {MODEL_KEY} --key {MODEL_KEY}_pilot \
    --langs ru --subsets Gender_identity --no_control \
    --work_dir {PILOT}/work --results_dir {PILOT}
pilot_secs = time.time() - t0
# `!` does not stop "Run all" on failure; this does.
assert globals().get("_exit_code", 0) == 0, "run failed: see the error above"
print(f"{pilot_secs / 3600:.2f} h, including model download and loading")

In [ ]:
# Quick look; the full analysis happens locally on the downloaded pickle.
import sys
sys.path.insert(0, "my_research/scorer")
from mbbq_scorer import annotate, attach_polarity, load_outputs, score

ru = load_outputs(f"{PILOT}/{MODEL_KEY}_pilot_ru.pkl")
undetected = ru[ru["answer_detected"].isna()]
print(f"ru Gender_identity: {len(ru)} rows, {len(undetected) / len(ru):.1%} undetected")
print("by prompt:", ru.groupby("prompt_id")["answer_detected"]
      .apply(lambda s: f"{s.isna().mean():.1%}").to_dict())
en_pkl = f"{DRIVE}/results/{MODEL_KEY}_en.pkl"
if os.path.exists(en_pkl):  # from the full English run
    en = load_outputs(en_pkl)
    en = en[en["subset"] == "Gender_identity"]
    print(f"en Gender_identity: {en['answer_detected'].isna().mean():.1%} undetected")

s = score(annotate(attach_polarity(ru, PILOT_DATA, "ru")))
print({k: round(v, 3) for k, v in s.items()})

for _, r in undetected.sample(min(10, len(undetected)), random_state=0).iterrows():
    print(f"\n--- prompt {r['prompt_id']}, options {r['model_ans']}\n{r['answer'][:500]}")

## Russian full run (step 5 in `NEXT_STEPS.md`)

One model per pass (`MODEL_KEY` in the first cell): `ru` main and control on all six subsets, then `ru_masc` (Gender_identity with the masculine-default questions) for the leak comparison.

| Set | Items | Prompts |
|---|---|---|
| `ru` | 10,088 | 50,440 |
| `ru` control | 10,072 | 50,360 |
| `ru_masc` | 152 | 760 |

That is about 37 times the pilot's 2,720 prompts: use the time the pilot cell printed to estimate the GPU hours.

Run the settings and setup cells, then the cells below; skip the smoke test, the full run and the pilot. **After a disconnect, run the same cells again:** finished subsets are skipped.

- **All 98 templates run,** whatever the vote keeps: the scorer drops the rejected ones afterwards, so the runs don't wait for V4–V7.
- **The data is generated once** into Drive (`ru_full/work/data`) and then stays fixed: the setup cell's `git pull` doesn't change it, so a worksheet edit can't mix two versions of the data in one run. To regenerate on purpose, delete that folder and every `trial*` file in `ru_full/work`.
- If a per-subset file doesn't match the data, `run_step6.py` stops before asking the model and names the file to delete.

When all three are done, download `ru_full/results/<model>_ru.pkl`, `<model>_ru_control.pkl` and `<model>_ru_masc.pkl` into `my_research/results/`.

In [ ]:
# Generate the Russian data into Drive once, and record the commit it came from.
import os, subprocess, time
RU = f"{DRIVE}/ru_full"
RU_DATA = f"{RU}/work/data"
STAMP = f"{RU_DATA}/GENERATED_FROM.txt"
if not os.path.exists(STAMP):
    os.makedirs(RU_DATA, exist_ok=True)
    !python my_research/templates/generate_ru.py --out {RU_DATA}
    assert globals().get("_exit_code", 0) == 0, "generation failed: see the error above"
    commit = subprocess.run(["git", "log", "-1", "--format=%h %cd", "--date=iso"],
                            capture_output=True, text=True).stdout.strip()
    with open(STAMP, "w") as f:
        f.write(commit + "\n")
print(f"Russian data in {RU_DATA}, generated from commit {open(STAMP).read().strip()}")

In [ ]:
# ru, main then control, all six subsets. Resumes at the unfinished subset.
t0 = time.time()
!python my_research/generation/run_step6.py --model {MODEL_KEY} --langs ru \
    --work_dir {RU}/work --results_dir {RU}/results
print(f"{(time.time() - t0) / 3600:.2f} h this session")
# `!` does not stop "Run all" on failure; this does.
assert globals().get("_exit_code", 0) == 0, "run failed: see the error above"

In [ ]:
# ru_masc: Gender_identity with the masculine-default questions, main set only.
!python my_research/generation/run_step6.py --model {MODEL_KEY} --langs ru_masc \
    --subsets Gender_identity --no_control --work_dir {RU}/work --results_dir {RU}/results
assert globals().get("_exit_code", 0) == 0, "run failed: see the error above"

In [ ]:
# Quick look at what is done; the full analysis happens locally.
import sys
sys.path.insert(0, "my_research/scorer")
from mbbq_scorer import annotate, attach_polarity, load_outputs, score

for tag, lang, control in [("ru", "ru", False), ("ru_control", "ru", True), ("ru_masc", "ru_masc", False)]:
    path = f"{RU}/results/{MODEL_KEY}_{tag}.pkl"
    if not os.path.exists(path):
        print(f"{tag}: not done yet")
        continue
    df = load_outputs(path)
    print(f"\n{tag}: {len(df)} rows, {df['answer_detected'].isna().mean():.2%} undetected; by subset:",
          df.groupby("subset")["answer_detected"].apply(lambda s: f"{s.isna().mean():.1%}").to_dict())
    s = score(annotate(attach_polarity(df, RU_DATA, lang, control=control)))
    print("  all templates, no filter:", {k: round(v, 3) for k, v in s.items()})

## Feminine variant (`DECISIONS.md`, "Feminine variant")

One model per pass (`MODEL_KEY` in the first cell). Gender_identity, main set only:

| Set | Items | Prompts | What |
|---|---|---|---|
| `ru_fem` | 144 | 720 | The `masc` questions with feminine agreement («Кто была секретарём?»), 17 templates |
| `ru_masc` rerun | 152 | 760 | The same `ru_masc` again, in the same session: the noise floor |

About 1,480 prompts, roughly half the pilot. Everything goes to `MyDrive/mbbq_paper/ru_fem`, so the full run's folder (`ru_full`) and its data stay as they were.

Run the settings and setup cells, then the two cells below; skip the other sections. **After a disconnect, run the same cells again:** finished subsets are skipped.

When both are done, download `ru_fem/results/<model>_ru_fem.pkl` and `<model>_ru_masc_rerun.pkl` into `my_research/results/` (not the plain `<model>_ru_masc.pkl` in that folder: it is the same rerun under the full run's name). Then run `python3 my_research/scorer/fem_analysis.py` locally.

In [ ]:
# Generate Gender_identity (main, control, masc, fem) into Drive once, then
# run ru_fem and rerun ru_masc. The model loads once for both.
import os, shutil, subprocess, time
assert MODEL_KEY in ("mistral", "zephyr"), "qwen and tlite: use the Replication models section"
FEM = f"{DRIVE}/ru_fem"
FEM_DATA = f"{FEM}/work/data"
FEM_STAMP = f"{FEM_DATA}/GENERATED_FROM.txt"
if not os.path.exists(FEM_STAMP):
    os.makedirs(FEM_DATA, exist_ok=True)
    !python my_research/templates/generate_ru.py --category Gender_identity --out {FEM_DATA}
    assert globals().get("_exit_code", 0) == 0, "generation failed: see the error above"
    commit = subprocess.run(["git", "log", "-1", "--format=%h %cd", "--date=iso"],
                            capture_output=True, text=True).stdout.strip()
    with open(FEM_STAMP, "w") as f:
        f.write(commit + "\n")
assert os.path.exists(f"{FEM_DATA}/Gender_identity_ru_fem.jsonl"), \
    "no ru_fem data: pull the commit that adds the fem rows, delete ru_fem/work and rerun"
print(f"Data in {FEM_DATA}, generated from commit {open(FEM_STAMP).read().strip()}")

t0 = time.time()
!python my_research/generation/run_step6.py --model {MODEL_KEY} --langs ru_fem ru_masc \
    --subsets Gender_identity --no_control --work_dir {FEM}/work --results_dir {FEM}/results
print(f"{(time.time() - t0) / 3600:.2f} h this session")
# `!` does not stop "Run all" on failure; this does.
assert globals().get("_exit_code", 0) == 0, "run failed: see the error above"

# The rerun gets its own name, so it can't be mistaken for the full run's ru_masc.
src, dst = f"{FEM}/results/{MODEL_KEY}_ru_masc.pkl", f"{FEM}/results/{MODEL_KEY}_ru_masc_rerun.pkl"
if os.path.exists(src) and not os.path.exists(dst):
    shutil.copy(src, dst)

In [ ]:
# Quick look; the analysis (fem_analysis.py) runs locally on the downloaded pickles.
import sys
sys.path.insert(0, "my_research/scorer")
from mbbq_scorer import annotate, attach_polarity, load_outputs, score

for tag, lang in [("ru_fem", "ru_fem"), ("ru_masc_rerun", "ru_masc")]:
    path = f"{FEM}/results/{MODEL_KEY}_{tag}.pkl"
    if not os.path.exists(path):
        print(f"{tag}: not done yet")
        continue
    df = load_outputs(path)
    print(f"\n{tag}: {len(df)} rows, {df['answer_detected'].isna().mean():.2%} undetected")
    s = score(annotate(attach_polarity(df, FEM_DATA, lang)))
    print("  all templates, no filter:", {k: round(v, 3) for k, v in s.items()})

## Replication models (`DECISIONS.md`, "Replication on two more models")

Qwen2.5-7B-Instruct (`MODEL_KEY = "qwen"`) and T-lite-it-1.0 (`MODEL_KEY = "tlite"`), one per pass. Both are open, so no Hugging Face token is needed. Settings come from `REPLICATION_MODELS` in `ask_model_paper.py`: each model's own chat template, greedy decoding, 512 new tokens, batch size 16.

Gender_identity, main set only:

| Set | Items | Prompts |
|---|---|---|
| `ru` | 544 | 2,720 |
| `ru_masc` | 152 | 760 |
| `ru_fem` | 144 | 720 |
| `ru_masc` rerun | 152 | 760 |

About 4,960 prompts per model. Everything goes to `MyDrive/mbbq_paper/ru_models`; the other folders aren't touched.

Set `MODEL_KEY` in the first cell, run the settings and setup cells, then the two cells below; skip the other sections. **After a disconnect, run the same cells again:** finished sets are skipped.

When both models are done, download from `ru_models/results/` into `my_research/results/`, for each model: `<model>_ru.pkl`, `<model>_ru_masc.pkl`, `<model>_ru_fem.pkl` and `<model>_ru_masc_rerun.pkl` (not `<model>_rerun_ru_masc.pkl`, the same file under the run's own name). Then run `python3 my_research/scorer/fem_analysis.py` and `fem_congruence.py` locally.

In [ ]:
# Generate Gender_identity into Drive once, then run ru, ru_masc and ru_fem, and
# rerun ru_masc under its own key (the noise floor).
import os, shutil, subprocess, time
assert MODEL_KEY in ("qwen", "tlite"), 'set MODEL_KEY = "qwen" or "tlite" in the first cell'
REP = f"{DRIVE}/ru_models"
REP_DATA = f"{REP}/work/data"
REP_STAMP = f"{REP_DATA}/GENERATED_FROM.txt"
if not os.path.exists(REP_STAMP):
    os.makedirs(REP_DATA, exist_ok=True)
    !python my_research/templates/generate_ru.py --category Gender_identity --out {REP_DATA}
    assert globals().get("_exit_code", 0) == 0, "generation failed: see the error above"
    commit = subprocess.run(["git", "log", "-1", "--format=%h %cd", "--date=iso"],
                            capture_output=True, text=True).stdout.strip()
    with open(REP_STAMP, "w") as f:
        f.write(commit + "\n")
assert os.path.exists(f"{REP_DATA}/Gender_identity_ru_fem.jsonl"), \
    "no ru_fem data: pull the latest commit, delete ru_models/work and rerun"
print(f"Data in {REP_DATA}, generated from commit {open(REP_STAMP).read().strip()}")

t0 = time.time()
!python my_research/generation/run_step6.py --model {MODEL_KEY} --langs ru ru_masc ru_fem \
    --subsets Gender_identity --no_control --work_dir {REP}/work --results_dir {REP}/results
assert globals().get("_exit_code", 0) == 0, "run failed: see the error above"
!python my_research/generation/run_step6.py --model {MODEL_KEY} --key {MODEL_KEY}_rerun \
    --langs ru_masc --subsets Gender_identity --no_control \
    --work_dir {REP}/work --results_dir {REP}/results
assert globals().get("_exit_code", 0) == 0, "rerun failed: see the error above"
print(f"{(time.time() - t0) / 3600:.2f} h this session")

src = f"{REP}/results/{MODEL_KEY}_rerun_ru_masc.pkl"
dst = f"{REP}/results/{MODEL_KEY}_ru_masc_rerun.pkl"
if os.path.exists(src) and not os.path.exists(dst):
    shutil.copy(src, dst)

In [ ]:
# Quick look; the analysis runs locally on the downloaded pickles.
import sys
sys.path.insert(0, "my_research/scorer")
from mbbq_scorer import annotate, attach_polarity, load_outputs, score

for tag, lang in [("ru", "ru"), ("ru_masc", "ru_masc"), ("ru_fem", "ru_fem"),
                  ("ru_masc_rerun", "ru_masc")]:
    path = f"{REP}/results/{MODEL_KEY}_{tag}.pkl"
    if not os.path.exists(path):
        print(f"{tag}: not done yet")
        continue
    df = load_outputs(path)
    print(f"\n{tag}: {len(df)} rows, {df['answer_detected'].isna().mean():.2%} undetected")
    print(df["answer"].sample(3, random_state=0).str[:200].tolist())
    s = score(annotate(attach_polarity(df, REP_DATA, lang)))
    print("  all templates, no filter:", {k: round(v, 3) for k, v in s.items()})

## Batch-size noise floor (`DECISIONS.md`, "Batch-size noise floor")

`ru_masc` once more for each of the four models, at **half its default batch size**: Mistral 8, Zephyr 16, Qwen 8, T-lite 8. Changing the batch size changes how prompts are padded and grouped, which is the noise a same-session rerun can't show. 760 prompts per model.

One model per pass: set `MODEL_KEY` in the first cell (`"mistral"`, `"zephyr"`, `"qwen"` or `"tlite"`; the transformers version follows from it), run the settings and setup cells, then the cell below. Restart the session between a paper model (Mistral, Zephyr) and a replication model (Qwen, T-lite), since they need different transformers versions. Everything goes to `MyDrive/mbbq_paper/noise_floor`.

When all four are done, download `noise_floor/results/<model>_ru_masc_batch.pkl` (four files) into `my_research/results/`, then run `fem_analysis.py` and `fem_congruence.py` locally.

In [ ]:
# ru_masc at half the default batch size, under its own key, in its own folder.
import os, shutil, subprocess, time
HALF_BS = {"mistral": 8, "zephyr": 16, "qwen": 8, "tlite": 8}  # defaults 16, 32, 16, 16
assert MODEL_KEY in HALF_BS, f"MODEL_KEY must be one of {sorted(HALF_BS)}"
BS = HALF_BS[MODEL_KEY]
NF = f"{DRIVE}/noise_floor"
NF_DATA = f"{NF}/work/data"
NF_STAMP = f"{NF_DATA}/GENERATED_FROM.txt"
if not os.path.exists(NF_STAMP):
    os.makedirs(NF_DATA, exist_ok=True)
    !python my_research/templates/generate_ru.py --category Gender_identity --out {NF_DATA}
    assert globals().get("_exit_code", 0) == 0, "generation failed: see the error above"
    commit = subprocess.run(["git", "log", "-1", "--format=%h %cd", "--date=iso"],
                            capture_output=True, text=True).stdout.strip()
    with open(NF_STAMP, "w") as f:
        f.write(commit + "\n")
print(f"Data in {NF_DATA}, generated from commit {open(NF_STAMP).read().strip()}")

t0 = time.time()
!python my_research/generation/run_step6.py --model {MODEL_KEY} --key {MODEL_KEY}_bs{BS} --bs {BS} \
    --langs ru_masc --subsets Gender_identity --no_control \
    --work_dir {NF}/work --results_dir {NF}/results
print(f"{(time.time() - t0) / 3600:.2f} h this session")
# `!` does not stop "Run all" on failure; this does.
assert globals().get("_exit_code", 0) == 0, "run failed: see the error above"

src = f"{NF}/results/{MODEL_KEY}_bs{BS}_ru_masc.pkl"
dst = f"{NF}/results/{MODEL_KEY}_ru_masc_batch.pkl"
if os.path.exists(src) and not os.path.exists(dst):
    shutil.copy(src, dst)
print("ready to download:", dst if os.path.exists(dst) else "not yet")

## Fifth model (`DECISIONS.md`, "Fifth model")

Aya-101 (`MODEL_KEY = "aya"`) first; Llama-2-7B-Chat (`"llama2"`) only if Aya fails the rule. Settings come from `FIFTH_MODELS` in `ask_model_paper.py` (MBBQ's own `models.py`): Aya with `text2text-generation`, the bare question, 128 new tokens and batch size 8; transformers 4.39.3, as for Mistral and Zephyr. Everything goes to `MyDrive/mbbq_paper/fifth_model`.

- **Aya-101** downloads 52 GB (stored in fp32) and loads as 26 GB in bf16: use an **A100**. The L4's 24 GB is too small.
- **Llama-2** is gated: request access on its Hugging Face page and add the `HF_TOKEN` secret before running it.

**Step 1, pilot (neutral set only):** set `MODEL_KEY`, run the settings and setup cells, then the pilot cell. Download `fifth_model/results/<model>_ru.pkl` into `my_research/results/` and run `python3 my_research/scorer/fifth_model_pilot.py <model>` locally. **Don't run step 2 unless the model passes.**

**Step 2, only after it passes:** the second cell runs `ru_masc`, `ru_fem`, a `ru_masc` rerun and the batch-size floor. Download `<model>_ru_masc.pkl`, `<model>_ru_fem.pkl`, `<model>_ru_masc_rerun.pkl` and `<model>_ru_masc_batch.pkl`.

In [ ]:
# Step 1: the pilot. ru, Gender_identity, main set (2,720 prompts); no masc or fem.
import os, shutil, subprocess, time
assert MODEL_KEY in ("aya", "llama2"), 'set MODEL_KEY = "aya" (or "llama2") in the first cell'
FIFTH = f"{DRIVE}/fifth_model"
FIFTH_DATA = f"{FIFTH}/work/data"
FIFTH_STAMP = f"{FIFTH_DATA}/GENERATED_FROM.txt"
if not os.path.exists(FIFTH_STAMP):
    os.makedirs(FIFTH_DATA, exist_ok=True)
    !python my_research/templates/generate_ru.py --category Gender_identity --out {FIFTH_DATA}
    assert globals().get("_exit_code", 0) == 0, "generation failed: see the error above"
    commit = subprocess.run(["git", "log", "-1", "--format=%h %cd", "--date=iso"],
                            capture_output=True, text=True).stdout.strip()
    with open(FIFTH_STAMP, "w") as f:
        f.write(commit + "\n")
print(f"Data in {FIFTH_DATA}, generated from commit {open(FIFTH_STAMP).read().strip()}")

t0 = time.time()
!python my_research/generation/run_step6.py --model {MODEL_KEY} --langs ru \
    --subsets Gender_identity --no_control --work_dir {FIFTH}/work --results_dir {FIFTH}/results
print(f"{(time.time() - t0) / 3600:.2f} h this session, including download and loading")
# `!` does not stop "Run all" on failure; this does.
assert globals().get("_exit_code", 0) == 0, "run failed: see the error above"

import sys
sys.path.insert(0, "my_research/scorer")
from mbbq_scorer import load_outputs
df = load_outputs(f"{FIFTH}/results/{MODEL_KEY}_ru.pkl")
print(f"{len(df)} rows, {df['answer_detected'].isna().mean():.2%} undetected (stored detection)")
print(df["answer"].sample(5, random_state=0).str[:200].tolist())
print("Now download it and run fifth_model_pilot.py locally: the rule decides, not this cell.")

In [ ]:
# Step 2, ONLY after fifth_model_pilot.py says the model passes:
# ru_masc, ru_fem, the ru_masc rerun (noise floor) and ru_masc at half the batch size.
assert MODEL_KEY in ("aya", "llama2")
FIFTH = f"{DRIVE}/fifth_model"
HALF_BS = {"aya": 4, "llama2": 2}  # defaults 8, 4
t0 = time.time()
!python my_research/generation/run_step6.py --model {MODEL_KEY} --langs ru_masc ru_fem \
    --subsets Gender_identity --no_control --work_dir {FIFTH}/work --results_dir {FIFTH}/results
assert globals().get("_exit_code", 0) == 0, "run failed: see the error above"
!python my_research/generation/run_step6.py --model {MODEL_KEY} --key {MODEL_KEY}_rerun \
    --langs ru_masc --subsets Gender_identity --no_control \
    --work_dir {FIFTH}/work --results_dir {FIFTH}/results
assert globals().get("_exit_code", 0) == 0, "rerun failed: see the error above"
BS = HALF_BS[MODEL_KEY]
!python my_research/generation/run_step6.py --model {MODEL_KEY} --key {MODEL_KEY}_bs{BS} --bs {BS} \
    --langs ru_masc --subsets Gender_identity --no_control \
    --work_dir {FIFTH}/work --results_dir {FIFTH}/results
assert globals().get("_exit_code", 0) == 0, "batch-size run failed: see the error above"
print(f"{(time.time() - t0) / 3600:.2f} h this session")

for src, dst in [(f"{MODEL_KEY}_rerun_ru_masc.pkl", f"{MODEL_KEY}_ru_masc_rerun.pkl"),
                 (f"{MODEL_KEY}_bs{BS}_ru_masc.pkl", f"{MODEL_KEY}_ru_masc_batch.pkl")]:
    src, dst = f"{FIFTH}/results/{src}", f"{FIFTH}/results/{dst}"
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst)
print(sorted(os.listdir(f"{FIFTH}/results")))